Lab08

Bai01

In [1]:
# -*- coding: utf-8 -*-
"""Buoc 1: tu viet K-Means bang numpy, chay tren bay diem cua slide."""
import numpy as np
from sklearn.cluster import KMeans

P = np.array([
    [1.0, 1.0], [1.5, 2.0], [3.0, 4.0], [5.0, 7.0],
    [3.5, 5.0], [4.5, 5.0], [3.5, 4.5],
])
TEN = ["A", "B", "C", "D", "E", "F", "G"]


def k_means(X, tam, so_vong_toi_da=10):
    """K-Means don gian: gan diem vao tam gan nhat roi doi tam."""
    nhan = None
    for vong in range(1, so_vong_toi_da + 1):
        # Buoc gan: khoang cach tu moi diem toi moi tam
        kc = np.sqrt(((X[:, None, :] - tam[None, :, :]) ** 2).sum(axis=2))
        nhan = kc.argmin(axis=1)
        print(f"Vong {vong}: tam = {np.round(tam, 2).tolist()}")
        for k in range(len(tam)):
            ds = [TEN[i] for i in range(len(X)) if nhan[i] == k]
            print(f" cum {k + 1}: {', '.join(ds)}")
        # Buoc doi tam: trung binh cac diem trong moi cum
        tam_moi = np.array([
            X[nhan == k].mean(axis=0) for k in range(len(tam))
        ])
        if np.allclose(tam_moi, tam):
            print(" Tam khong doi nua, dung.")
            return tam, nhan
        tam = tam_moi
    return tam, nhan


# Khoi tao co y te: hai tam la A va B, nam sat nhau
tam, nhan = k_means(P, P[[0, 1]].copy())
print()

inertia = ((P - tam[nhan]) ** 2).sum()
print(f"Inertia tu tinh: {inertia:.4f}")

km = KMeans(n_clusters=2, init=P[[0, 1]], n_init=1, random_state=0).fit(P)
print(f"Inertia cua scikit-learn: {km.inertia_:.4f}")
print("Tam cua scikit-learn:", np.round(km.cluster_centers_, 2).tolist())

Vong 1: tam = [[1.0, 1.0], [1.5, 2.0]]
 cum 1: A
 cum 2: B, C, D, E, F, G
Vong 2: tam = [[1.0, 1.0], [3.5, 4.58]]
 cum 1: A, B
 cum 2: C, D, E, F, G
Vong 3: tam = [[1.25, 1.5], [3.9, 5.1]]
 cum 1: A, B
 cum 2: C, D, E, F, G
 Tam khong doi nua, dung.

Inertia tu tinh: 8.5250
Inertia cua scikit-learn: 8.5250
Tam cua scikit-learn: [[1.25, 1.5], [3.9, 5.1]]


Bai02

In [2]:
# -*- coding: utf-8 -*-
"""Buoc 2: phan khuc khach hang bang K-Means."""
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
print("Kich thuoc bang:", df.shape)
print(df.describe().round(1).to_string())
print()

X = df[["thu_nhap", "diem_chi_tieu"]]
sc = StandardScaler().fit(X)
Xs = sc.transform(X)

km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
df["cum"] = km.labels_

# Tam cum dang o thang chuan hoa, doi nguoc ve don vi goc
tam = pd.DataFrame(sc.inverse_transform(km.cluster_centers_),
                   columns=["thu_nhap", "diem_chi_tieu"])
tam["so_khach"] = np.bincount(km.labels_)
tam["tuoi_tb"] = df.groupby("cum")["tuoi"].mean()
print("Tam cum, don vi goc:")
print(tam.round(1).to_string())
print()
print(f"Inertia: {km.inertia_:.2f}")
print()

# Khach moi: chuan hoa bang dung bo sc cu, roi tim tam gan nhat
moi = pd.DataFrame({"thu_nhap": [29.0, 9.0],
                    "diem_chi_tieu": [85, 75]})
cum_moi = km.predict(sc.transform(moi))
for i in range(len(moi)):
    print(f"Khach moi {i + 1}: thu nhap {moi.iloc[i, 0]}, "
          f"chi tieu {moi.iloc[i, 1]} -> cum {cum_moi[i]}")

Kich thuoc bang: (210, 3)
        tuoi  thu_nhap  diem_chi_tieu
count  210.0     210.0          210.0
mean    37.6      18.5           49.9
std     10.3       9.8           26.8
min     18.0       3.0            7.0
25%     30.0       8.8           22.0
50%     39.0      17.4           49.0
75%     46.0      28.6           77.0
max     62.0      35.5           99.0

Tam cum, don vi goc:
   thu_nhap  diem_chi_tieu  so_khach  tuoi_tb
0       7.5           80.2        35     24.9
1      30.1           20.0        35     42.9
2      17.4           49.3        60     40.4
3       8.0           18.9        40     44.9
4      30.2           81.5        40     32.6

Inertia: 23.02

Khach moi 1: thu nhap 29.0, chi tieu 85 -> cum 4
Khach moi 2: thu nhap 9.0, chi tieu 75 -> cum 0


Bai03

In [3]:
# -*- coding: utf-8 -*-
"""Buoc 3: chon so cum K bang khuyu tay va chi so silhouette."""
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
Xs = StandardScaler().fit_transform(df[["thu_nhap", "diem_chi_tieu"]])

print(" K  inertia  giam so voi K-1  silhouette")
truoc = None
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    giam = "" if truoc is None else f"{truoc - km.inertia_:8.1f}"
    sil = "" if k == 1 else f"{silhouette_score(Xs, km.labels_):.3f}"
    print(f" {k:2d} {km.inertia_:7.1f} {giam:>17s} {sil:>10s}")
    truoc = km.inertia_

 K  inertia  giam so voi K-1  silhouette
  1   420.0                             
  2   250.4             169.6      0.430
  3   152.5              97.9      0.522
  4    73.8              78.8      0.626
  5    23.0              50.7      0.723
  6    20.9               2.1      0.651
  7    19.0               1.9      0.541
  8    17.4               1.6      0.454
  9    15.9               1.5      0.471
 10    14.4               1.4      0.425


bai04

In [5]:
# -*- coding: utf-8 -*-
"""Buoc 4: ba cai bay cua K-Means."""
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
X = df[["thu_nhap", "diem_chi_tieu"]]
Xs = StandardScaler().fit_transform(X)

# ----- Bay 1: thu nhap ghi bang nghin dong, khong chuan hoa -----
X_nghin = X.copy()
X_nghin["thu_nhap"] = X_nghin["thu_nhap"] * 1000
km_tho = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X_nghin)
tam_tho = km_tho.cluster_centers_.copy()
tam_tho[:, 0] = tam_tho[:, 0] / 1000   # doi lai trieu de de doc
print("Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):")
for t in tam_tho[np.argsort(tam_tho[:, 0])]:
    print(f"  ({t[0]:5.1f}, {t[1]:5.1f})")
print()

# ----- Bay 2: moi lan chi khoi tao ngau nhien mot lan -----
ket = []
for rs in range(20):
    km = KMeans(n_clusters=5, init="random", n_init=1,
                random_state=rs).fit(Xs)
    ket.append(round(km.inertia_, 1))
print("Bay 2: inertia cua 20 lan khoi tao ngau nhien:")
ket = sorted(ket)
print("  ", ket[:10])
print("  ", ket[10:])
tot = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
print(f" Voi n_init=10: inertia {tot.inertia_:.1f}")
print()

# ----- Bay 3: cum khong tron, du lieu hai vang trang -----
tr = pd.read_csv("hai_vang_trang.csv")
km_tr = KMeans(n_clusters=2, n_init=10, random_state=0)
nhan = km_tr.fit_predict(tr[["x1", "x2"]])
khop = (nhan == tr["nhan"]).mean()
print(f"Bay 3: hai vang trang, ty le khop nhan that "
      f"{max(khop, 1 - khop):.4f}")

Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):
  (  6.1,  52.5)
  (  9.6,  41.9)
  ( 17.4,  49.8)
  ( 27.8,  52.9)
  ( 31.9,  52.4)

Bay 2: inertia cua 20 lan khoi tao ngau nhien:
   [23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0]
   [72.4, 73.6, 73.7, 73.7, 84.0, 84.2, 84.2, 116.6, 117.4, 118.2]
 Voi n_init=10: inertia 23.0

Bay 3: hai vang trang, ty le khop nhan that 0.7500


Bai05

In [6]:
# -*- coding: utf-8 -*-
"""Buoc 5: dung K-Means de nen mau mot buc anh."""
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image
from sklearn.utils import shuffle

# Anh mau co san trong scikit-learn, khong can tai them
china = load_sample_image("china.jpg")
print("Kich thuoc anh:", china.shape)

# Doi ve [0, 1] roi dua ve mang 2 chieu (so diem anh, 3 kenh mau)
china = np.array(china, dtype=np.float64) / 255.0
w, h, d = china.shape
X = china.reshape(w * h, d)
print("So diem anh:", X.shape[0])

# Lay mau 1000 diem de fit nhanh
X_mau = shuffle(X, random_state=0)[:1000]

ket_qua = {}
for k in [2, 4, 8, 16, 32, 64]:
    t0 = time.time()
    km = KMeans(n_clusters=k, n_init=3, random_state=0).fit(X_mau)
    nhan = km.predict(X)
    X_nen = km.cluster_centers_[nhan]
    anh_nen = X_nen.reshape(w, h, d)
    sai_so = np.sqrt(((X - X_nen) ** 2).sum(axis=1)).mean()
    giay = time.time() - t0
    ket_qua[k] = (anh_nen, sai_so, giay)
    print(f"K={k:2d}: sai so trung binh {sai_so:.4f}, "
          f"thoi gian {giay:.1f}s")

# Ve so sanh anh goc va cac muc K
cac_k = [2, 8, 32]
fig, axes = plt.subplots(1, len(cac_k) + 1, figsize=(4 * (len(cac_k) + 1), 3))
axes[0].imshow(china)
axes[0].set_title("Anh goc")
axes[0].axis("off")
for ax, k in zip(axes[1:], cac_k):
    ax.imshow(ket_qua[k][0])
    ax.set_title(f"K = {k} (sai so {ket_qua[k][1]:.3f})")
    ax.axis("off")
plt.tight_layout()
plt.savefig("nen_anh_so_sanh.png", dpi=120)
print("\nDa luu hinh: nen_anh_so_sanh.png")

Kich thuoc anh: (427, 640, 3)
So diem anh: 273280
K= 2: sai so trung binh 0.2085, thoi gian 0.0s
K= 4: sai so trung binh 0.1317, thoi gian 0.0s
K= 8: sai so trung binh 0.0859, thoi gian 0.0s
K=16: sai so trung binh 0.0650, thoi gian 0.0s
K=32: sai so trung binh 0.0483, thoi gian 0.1s
K=64: sai so trung binh 0.0376, thoi gian 0.1s

Da luu hinh: nen_anh_so_sanh.png


Bai06

In [8]:
# -*- coding: utf-8 -*-
"""Bai tong hop: ap dung day du quy trinh K-Means tren hai vang trang,
kiem tra bay 'cum khong tron' va so sanh voi K = 2, 3, 4."""
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

tr = pd.read_csv("hai_vang_trang.csv")
print("Kich thuoc:", tr.shape)
print(tr.head())
print()

X = tr[["x1", "x2"]].to_numpy()
y_that = tr["nhan"].to_numpy()
Xs = StandardScaler().fit_transform(X)

# 1) Khuyu tay + silhouette
print(" K  inertia  giam so voi K-1  silhouette")
truoc = None
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    giam = "" if truoc is None else f"{truoc - km.inertia_:9.1f}"
    sil = "" if k == 1 else f"{silhouette_score(Xs, km.labels_):.3f}"
    print(f" {k:2d} {km.inertia_:7.1f} {giam:>15s} {sil:>10s}")
    truoc = km.inertia_
print()

# 2) K = 2 va do khop voi nhan that (co the am ban dao)
km2 = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xs)
nhan2 = km2.labels_
khop2 = max((nhan2 == y_that).mean(), (nhan2 != y_that).mean())
print(f"K = 2: ty le khop nhan that (tot nhat co the) = {khop2:.4f}")

# 3) K = 4: chia nho tung vang trăng
km4 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Xs)
print(f"K = 4: inertia {km4.inertia_:.2f}, "
      f"silhouette {silhouette_score(Xs, km4.labels_):.3f}")

# 4) Ve so sanh
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (ten, nhan) in zip(
    axes,
    [("Nhan that", y_that),
     ("K-Means K=2", km2.labels_),
     ("K-Means K=4", km4.labels_)],
):
    ax.scatter(X[:, 0], X[:, 1], c=nhan, cmap="viridis", s=15)
    ax.set_title(ten)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
plt.tight_layout()
plt.savefig("hai_vang_trang_kmeans.png", dpi=120)
print("\nDa luu hinh: hai_vang_trang_kmeans.png")

Kich thuoc: (300, 3)
       x1      x2  nhan
0 -0.2258  1.0332     0
1  0.7863 -0.5932     1
2  0.7073 -0.3852     1
3  2.1815  0.6666     1
4  0.9643  0.3850     0

 K  inertia  giam so voi K-1  silhouette
  1   600.0                           
  2   288.1           311.9      0.446
  3   218.0            70.1      0.396
  4   159.1            58.9      0.363
  5   121.9            37.2      0.383
  6   101.6            20.3      0.387
  7    85.6            16.0      0.376
  8    76.3             9.3      0.370
  9    68.8             7.5      0.350
 10    64.0             4.8      0.338

K = 2: ty le khop nhan that (tot nhat co the) = 0.8300
K = 4: inertia 159.07, silhouette 0.363

Da luu hinh: hai_vang_trang_kmeans.png
